# Imports

In [1]:
import os
import random
import numpy as np
import torch
import pandas as pd
from dqn_agent2 import DQNAgent
from ppo_agent2 import PPOAgent
from ffnn_agent2 import FFNNAgent
from lstm_agent import LSTMAgent
from data_processing import load_preprocessed_dataset, get_xy_from_data
from training import train_agents, train_model_baseline, evaluate_model
from visualize import plot_rewards, plot_episode_rewards, plot_cumulative_rewards, plot_mse_hist, plot_female_mse_hist, plot_episode_rewards_combined, plot_cumulative_rewards_combined

# Experiment Config

In [2]:
seed = 123
shuffle = False

accuracy_reward_multiplier = 10
synthetic_data_amount = 10
num_episodes = 10
batch_size = 32

experiment_name = "experiment_1"
results_folder = "latest_results"

# Load Data

In [3]:
# Load the preprocessed dataset
df = load_preprocessed_dataset()
print(df.head())


activity_split = 2
val_test = df[df['Activity ID'] <= activity_split]

val_test_female = val_test[val_test['Sex - Female'] == 1]
val_test_male   = val_test[val_test['Sex - Female'] == 0]

def split_half(df):
    n_val = int(len(df)*0.5)
    val_df  = df.iloc[:n_val].copy()   
    test_df = df.iloc[n_val:].copy()
    return val_df, test_df

female_val, female_test = split_half(val_test_female)
male_val, male_test = split_half(val_test_male)


# Step 4: Combine validation and test sets (sex-balanced)
df_val = pd.concat([ female_val, male_val], ignore_index=True).reset_index(drop=True)

df_test = pd.concat([female_test, male_test], ignore_index=True).reset_index(drop=True)

# Step 5: Training data = everything else
df_train = df[df['Activity ID'] > activity_split].copy()
df_train.reset_index(drop=True,inplace=True)

# Shuffle train set
if shuffle:
    df_train = df_train.sample(frac=1, random_state=seed).reset_index(drop=True)

# Report split sizes
print(f"\nTrain samples (excluding Activity 1 & 2): {df_train.shape[0]}")
print(f"Validation samples (Activity 1): {df_val.shape[0]}")
print(f"Test samples (Activity 2): {df_test.shape[0]}")

---- Preprocessed Dataset Info ----
Number of rows: 61000
Number of columns: 39
-----------------------------------
   Timestamp  Activity ID  Heart Rate  Hand Sensor - Temperature  \
0          0            1         100                     30.375   
1          1            1         100                     30.375   
2          2            1         100                     30.375   
3          3            1         100                     30.375   
4          4            1         101                     30.375   

   Hand Sensor - Accelerometer - X  Hand Sensor - Accelerometer - Y  \
0                          2.30106                          7.25857   
1                          2.24615                          7.48180   
2                          2.30000                          7.10681   
3                          2.49455                          7.52335   
4                          2.71654                          8.30596   

   Hand Sensor - Accelerometer - Z  Hand Sensor 

# Initialize system

In [4]:
random.seed(seed)
np.random.seed(seed)

target_features = ["Sex - Female", "Heart Rate", "Age", "Weight", "Height"]

#To be generated
continuous_columns = [
    'Hand Sensor - Temperature',
    'Hand Sensor - Accelerometer - X', 'Hand Sensor - Accelerometer - Y',
    'Hand Sensor - Accelerometer - Z', 'Hand Sensor - Gyroscope - X',
    'Hand Sensor - Gyroscope - Y', 'Hand Sensor - Gyroscope - Z',
    'Hand Sensor - Magnetometer - X', 'Hand Sensor - Magnetometer - Y',
    'Hand Sensor - Magnetometer - Z', 'Chest Sensor - Temperature',
    'Chest Sensor - Accelerometer - X', 'Chest Sensor - Accelerometer - Y',
    'Chest Sensor - Accelerometer - Z', 'Chest Sensor - Gyroscope - X',
    'Chest Sensor - Gyroscope - Y', 'Chest Sensor - Gyroscope - Z',
    'Chest Sensor - Magnetometer - X', 'Chest Sensor - Magnetometer - Y',
    'Chest Sensor - Magnetometer - Z', 'Ankle Sensor - Temperature',
    'Ankle Sensor - Accelerometer - X', 'Ankle Sensor - Accelerometer - Y',
    'Ankle Sensor - Accelerometer - Z', 'Ankle Sensor - Gyroscope - X',
    'Ankle Sensor - Gyroscope - Y', 'Ankle Sensor - Gyroscope - Z',
    'Ankle Sensor - Magnetometer - X', 'Ankle Sensor - Magnetometer - Y',
    'Ankle Sensor - Magnetometer - Z'
]
print(f"Number of continuous columns: {len(continuous_columns)}")  # 30

discrete_columns = [
    "Sex - Female", 'Heart Rate', "Resting HR", "Max HR", "Weight", "Height"
]
print(f"Number of discrete columns: {len(discrete_columns)}")  # 6

#discrete action size columns
dqn_config = {
    'state_size': 5,  
    'action_size': len(discrete_columns),  
    'hidden_size': 64,
    'lr': 1e-2, # HYPERPARAMETER FOR EXPERIMENTS
    'gamma': 0.8, # HYPERPARAMETER FOR EXPERIMENTS
    'batch_size': batch_size,
    'memory_size': 10000,
    'epsilon_start': 1.0,
    'epsilon_min': 0.01,
    'epsilon_decay': 0.995,
    'seed': seed
}

#continuous
ppo_config = {
    'state_size': 5,  
    'action_size': len(continuous_columns),   
    'hidden_size': 64,
    'lr': 1e-2, # HYPERPARAMETER FOR EXPERIMENTS
    'gamma': 0.8, # HYPERPARAMETER FOR EXPERIMENTS
    'clip_epsilon': 0.2,
    'update_epochs': 10,
    'batch_size': batch_size,
    'c1': 0.5,
    'c2': 0.01,
    'seed': seed
}
classes = [1, 2, 3, 17, 16, 13, 4, 7, 6]

ffnn_config = { # DO NOT CHANGE THIS CONFIG
    'input_size': df.shape[1] - 5,
    'hidden_sizes': [16, 16],
    'output_size': 5,
    'learning_rate': 0.001,
    'batch_size': batch_size,
    'epochs': 10,
    'type': 'regression',
    'classes': None,
    'seed': seed
}

lstm_config = {
    'input_dim': df.shape[1] - 5,  # input features (excluding targets)
    'hidden_dim': 64,
    'output_dim': 5,               # number of target features
    'num_layers': 3,
    'bidirectional': True,
    'learning_rate': 0.001,
    'batch_size': batch_size,
    'epochs': 100,
    'seed': seed
}

gpu_id = 0
device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")

dqn_config["device"]  = device
ppo_config["device"]  = device
ffnn_config["device"] = device
lstm_config["device"] = device

dqn_agent = DQNAgent(**dqn_config)
ppo_agent = PPOAgent(**ppo_config)
ffnn_agent = FFNNAgent(**ffnn_config)
ffnn_agent_og = FFNNAgent(**ffnn_config)
lstm_agent = LSTMAgent(**lstm_config)

#Split dependent on target_features
x_train_df, y_train_df = get_xy_from_data(df_train, target_features)
x_val_df,   y_val_df   = get_xy_from_data(df_val,   target_features)
x_test_df,  y_test_df  = get_xy_from_data(df_test,  target_features)

print(x_train_df.head())

print("===================")

print(y_train_df.head())

Number of continuous columns: 30
Number of discrete columns: 6
FFNN Using device: cuda:0
FFNN Using device: cuda:0
   Timestamp  Activity ID  Hand Sensor - Temperature  \
0          0            3                    32.6875   
1          1            3                    32.6875   
2          2            3                    32.6875   
3          3            3                    32.6875   
4          4            3                    32.6875   

   Hand Sensor - Accelerometer - X  Hand Sensor - Accelerometer - Y  \
0                          2.58185                          8.82193   
1                          2.35965                          9.08992   
2                          2.29290                          8.82761   
3                          2.56785                          9.35840   
4                          2.70781                          7.76378   

   Hand Sensor - Accelerometer - Z  Hand Sensor - Gyroscope - X  \
0                          2.97405                    

# Training

In [5]:
# train agents
save_path = os.path.join(results_folder, experiment_name, "metrics")
os.makedirs(save_path, exist_ok=True)

# baseline_results = train_model_baseline(
#     ffnn_agent_og, df_train, df_val, df_test, target_features,
#     save_path, show_loss_plots=False, seed=seed
# )

training_results = train_agents(
    df_train, df_val, df_test, target_features,
    dqn_agent, ppo_agent, ffnn_agent, 
    continuous_columns, num_episodes, synthetic_data_amount, accuracy_reward_multiplier, 
    save_path, show_loss_plots=False, seed=seed
)

# save trained models
save_path = os.path.join(results_folder, experiment_name, "saved_models")
os.makedirs(save_path, exist_ok=True)
dqn_agent.save(os.path.join(save_path, "dqn_trained_model.pth"))
ppo_agent.save(os.path.join(save_path, "ppo_trained_model.pth"))

Episode 1/10: Generating Synthetic Data
Episode 1/10: Training FFNNAgent 
Episode 1/10: Evaluating FFNNAgent 
mini reward: 2.583455801010132
Episode 1/10 | Reward: -2861.9206
Val MSE: 286.4504 | Val Female MSE: 280.7774
Episode 1/10 completed in 18.84 seconds.

--------------------------------

Episode 2/10: Generating Synthetic Data
Episode 2/10: Training FFNNAgent 


KeyboardInterrupt: 

In [ ]:
# train agents
save_path = os.path.join(results_folder, experiment_name, "metrics")
os.makedirs(save_path, exist_ok=True)

baseline_results = train_model_baseline(
    ffnn_agent, df_train, df_val, df_test, target_features,
    save_path, show_loss_plots=False, seed=seed
)

training_results = train_agents(
    df_train, df_val, df_test, target_features,
    dqn_agent, ppo_agent, ffnn_agent, 
    continuous_columns, num_episodes, synthetic_data_amount, accuracy_reward_multiplier, 
    save_path, show_loss_plots=False, seed=seed
)

# save trained models
save_path = os.path.join(results_folder, experiment_name, "saved_models")
os.makedirs(save_path, exist_ok=True)
dqn_agent.save(os.path.join(save_path, "dqn_trained_model.pth"))
ppo_agent.save(os.path.join(save_path, "ppo_trained_model.pth"))

In [ ]:
# train agents
save_path = os.path.join(results_folder, experiment_name, "metrics")
os.makedirs(save_path, exist_ok=True)

# baseline_results = train_ffnn_baseline(
#     ffnn_agent_og, df_train, df_val, df_test, target_features,
#     save_path, show_loss_plots=False, seed=seed
# )

training_results = train_agents(
    df_train, df_val, df_test, target_features,
    dqn_agent, ppo_agent, ffnn_agent, 
    continuous_columns, num_episodes, synthetic_data_amount, accuracy_reward_multiplier, 
    save_path, show_loss_plots=False, seed=seed
)

# save trained models
save_path = os.path.join(results_folder, experiment_name, "saved_models")
os.makedirs(save_path, exist_ok=True)
dqn_agent.save(os.path.join(save_path, "dqn_trained_model.pth"))
ppo_agent.save(os.path.join(save_path, "ppo_trained_model.pth"))

# Testing


In [ ]:
x_test, y_test = get_xy_from_data(df_test, target_features)
mse, mae, female_mse = evaluate_model(ffnn_agent, x_test, y_test)
print("\n========== Trained FFNN Agent Test Results ============")
print(f"MSE: {mse:.4f} | MAE: {mae:.4f} | Female MSE: {female_mse:.4f}")
print("=======================================================\n")

# Visualize results

In [6]:
experiment_name = "experiment_1"
results_folder = "latest_results"

##### Latest Experiment

In [ ]:
# Plotting Nick's Experiment results
training_metrics_path = os.path.join(results_folder, experiment_name, 'metrics', 'training_metrics.json')
baseline_metrics_path = os.path.join(results_folder, experiment_name, 'metrics', 'baseline_metrics.json')
save_path = os.path.join(results_folder_name, experiment_name, 'plots')
os.makedirs(save_path, exist_ok=True)

plot_rewards(training_metrics_path, save_path)
plot_episode_rewards(training_metrics_path, save_path)
plot_cumulative_rewards(training_metrics_path, save_path)
plot_mse_hist(baseline_metrics_path, training_metrics_path, save_path)
plot_female_mse_hist(baseline_metrics_path, training_metrics_path, save_path)

##### Nick's Experiment
##### Hyperparameters: lr=1e-4, gamma=0.9995

In [ ]:
# Plotting Nick's Experiment results
training_metrics_path = os.path.join(results_folder_name, 'nick', 'metrics', 'training_metrics_nick.json')
baseline_metrics_path = os.path.join(results_folder_name, 'nick', 'metrics', 'baseline_metrics_nick.json')
save_path = os.path.join(results_folder_name, 'nick', 'plots')
os.makedirs(save_path, exist_ok=True)

plot_rewards(training_metrics_path, save_path)
plot_episode_rewards(training_metrics_path, save_path)
plot_cumulative_rewards(training_metrics_path, save_path)
plot_mse_hist(baseline_metrics_path, training_metrics_path, save_path)
plot_female_mse_hist(baseline_metrics_path, training_metrics_path, save_path)

##### Ethan's Experiment
##### Hyperparameters: lr=1e-3, gamma=0.99

In [ ]:
# Plotting Ethan's Experiment results
training_metrics_path = os.path.join(results_folder_name, 'ethan', 'metrics', 'training_metrics_ethan.json')
baseline_metrics_path = os.path.join(results_folder_name, 'ethan', 'metrics', 'baseline_metrics_ethan.json')
save_path = os.path.join(results_folder_name, 'ethan', 'plots')
os.makedirs(save_path, exist_ok=True)

plot_rewards(training_metrics_path, save_path)
plot_episode_rewards(training_metrics_path, save_path)
plot_cumulative_rewards(training_metrics_path, save_path)
plot_mse_hist(baseline_metrics_path, training_metrics_path, save_path)
plot_female_mse_hist(baseline_metrics_path, training_metrics_path, save_path)

##### Nadine's Experiment
##### Hyperparameters: lr=1e-2, gamma=0.8

In [ ]:
# Plotting Nadine's Experiment results
training_metrics_path = os.path.join(results_folder_name, 'nadine', 'metrics', 'training_metrics_nadine.json')
baseline_metrics_path = os.path.join(results_folder_name, 'nadine', 'metrics', 'baseline_metrics_nadine.json')
save_path = os.path.join(results_folder_name, 'nadine', 'plots')
os.makedirs(save_path, exist_ok=True)

plot_rewards(training_metrics_path, save_path)
plot_episode_rewards(training_metrics_path, save_path)
plot_cumulative_rewards(training_metrics_path, save_path)
plot_mse_hist(baseline_metrics_path, training_metrics_path, save_path)
plot_female_mse_hist(baseline_metrics_path, training_metrics_path, save_path)

##### Plotting Combined Reward Plots

In [ ]:
# Plotting combined results for all experiments
metrics_path_list = [
    os.path.join(results_folder_name, 'nick', 'metrics', 'training_metrics_nick.json'), 
    os.path.join(results_folder_name, 'ethan', 'metrics', 'training_metrics_ethan.json'), 
    os.path.join(results_folder_name, 'nadine', 'metrics', 'training_metrics_nadine.json')
]
experiment_hyperparams_list = [
    'lr=1e-4, gamma=0.9995',
    'lr=1e-3, gamma=0.99',
    'lr=1e-2, gamma=0.8'
]
save_path = os.path.join(results_folder_name, 'comparison_plots')
os.makedirs(save_path, exist_ok=True)

plot_episode_rewards_combined(metrics_path_list, experiment_hyperparams_list, save_path)
plot_cumulative_rewards_combined(metrics_path_list, experiment_hyperparams_list, save_path)